# 1. Data Collection (YouTube Data API)

Collect metadata and view counts for ~8,500 videos from 30 popular Indian YouTube channels.

**Why the official API instead of scraping?** YouTube's terms don't allow scraping, and scraped pages break whenever the layout changes.
The [YouTube Data API v3](https://developers.google.com/youtube/v3) is allowed, free, and returns clean, structured data (exact view counts, exact upload times).

**Quota.** The free tier allows 10,000 units per day. Searching costs 100 units per call, so this notebook avoids search entirely:

| Call | Cost | Used for |
|---|---|---|
| `channels.list` | 1 unit (up to 50 channels) | channel info + uploads playlist |
| `playlistItems.list` | 1 unit (50 videos) | list each channel's video IDs |
| `videos.list` | 1 unit (50 videos) | video details and view counts |

Total for this notebook: about **350 units**.

> This notebook calls the live API. Re-running it fetches today's view counts, so results will differ slightly from the saved CSVs.

## Setup

The API key is stored in a `.env` file (never in the code, and `.env` is in `.gitignore`).
Only whether the key loaded is printed, never the key itself, because notebook outputs are saved and pushed to GitHub.

In [4]:
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
import os

In [6]:
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
print("Key Loaded", API_KEY is not None)

Key Loaded True


In [9]:
youtube = build("youtube", "v3", developerKey=API_KEY)

## Channels

30 popular Indian channels across gaming, food, tech, education, music and entertainment.
Mixing niches is fine here because every video is only compared with **its own channel** (see `02_features.ipynb`).

In [11]:
CHANNEL_IDS = [
    "UCPjaNv8GlIamrxCbqincG_g",
    "UCX8pnu3DYUnx8qy8V_c6oHg",
    "UCPtPyb4VvaNqjkbL_tSVFVA",
    "UC0IWRLai-BAwci_e9MylNGw",
    "UC2QbAX5osa0F52s8dQIvqvQ",
    "UCF6FIQAAWhrmy6T1k_WZfGg",
    "UCYxMATvBqKQx7utYcYK3waA",
    "UCaA8TUWM6TJ5wj-DH60VEFg",
    "UCHpEfybj_wamBKz2u6IoiGA",
    "UCHciekxM6H6F3OTPrHs6B5g",
    "UCl_vAxZpvbO-PFXdDu7EdHw",
    "UCq-Fj5jknLsUf-MWSy4_brA",
    "UCOhHO2ICt0ti9KAh-QHvttQ",
    "UCEPL07qzVsOcHd3sMUws65g",
    "UCBwmMxybNva6P_5VmxjzwqA",
    "UCVOd6kA37ubwZvo0BZXKCMQ",
    "UC7Q7pl0z0MrdayvmAnchlJQ",
    "UCZXuz_mOCSadwft8wkzdrsw",
    "UCgcUFVynuKwaIZLJkjhzOVg",
    "UCVidr_Df7gMAcLB2HBe1ZOA",
    "UCcjk-KvDJBEvo8fJS29fekg",
    "UCZ9lCUhUOUrwqVJmfBkN92A",
    "UCrWdPsIgE4GIWGiFUMHL4bQ",
    "UC9MAhZQQd9egwWCxrwSIsJQ",
    "UC5y8sRxwKhWLyaP6Fxg_KHA",
    "UCCWi3hpnq_Pe03nGxuS7isg",
    "UCtvHtAA-6Kk_vbHYfYc909g",
    "UCbTOgDRbAA5qTQosHM7DjVg",
    "UCwA3yPBSbZpwse6Q0aA2LPg",
    "UCg1UcEEIfv8cX6Y4m67Icxw"
]

## Channel info and uploads playlists

Every channel has a hidden **uploads playlist** containing all its videos. Reading it costs 1 unit per 50 videos,
about 100× cheaper than searching. One `channels.list` call returns info for up to 50 channels.

In [14]:
response = youtube.channels().list(
    part="snippet,statistics,contentDetails",
    id=",".join(CHANNEL_IDS)
).execute()

channels = pd.DataFrame([{
    "channel_id": item["id"],
    "channel_name": item["snippet"]["title"],
    "subscribers": int(item["statistics"].get("subscriberCount", 0)),
    "uploads_playlist": item["contentDetails"]["relatedPlaylists"]["uploads"],
    "Number_of_videos": int(item["statistics"].get("videoCount", 0)),
} for item in response["items"]])

channels

,channel_id,channel_name,subscribers,uploads_playlist,Number_of_videos
0,UCgcUFVynuKwaIZLJkjhzOVg,MaddyEats,6410000,UUgcUFVynuKwaIZLJkjhzOVg,741
1,UC0IWRLai-BAwci_e9MylNGw,CarryisLive,12600000,UU0IWRLai-BAwci_e9MylNGw,1418
2,UC5y8sRxwKhWLyaP6Fxg_KHA,lastMinute,1760000,UU5y8sRxwKhWLyaP6Fxg_KHA,1932
3,UCPtPyb4VvaNqjkbL_tSVFVA,Thugesh,6640000,UUPtPyb4VvaNqjkbL_tSVFVA,501
4,UCYxMATvBqKQx7utYcYK3waA,Scout,5170000,UUYxMATvBqKQx7utYcYK3waA,1938
5,UCq-Fj5jknLsUf-MWSy4_brA,T-Series,315000000,UUq-Fj5jknLsUf-MWSy4_brA,27260
6,UCaA8TUWM6TJ5wj-DH60VEFg,RON GAMING,4580000,UUaA8TUWM6TJ5wj-DH60VEFg,2931
7,UC7Q7pl0z0MrdayvmAnchlJQ,MortaL,6950000,UU7Q7pl0z0MrdayvmAnchlJQ,2703
8,UCl_vAxZpvbO-PFXdDu7EdHw,Ujjwal,12700000,UUl_vAxZpvbO-PFXdDu7EdHw,461
9,UCrWdPsIgE4GIWGiFUMHL4bQ,Sumit 007,2480000,UUrWdPsIgE4GIWGiFUMHL4bQ,2108


## Video IDs (pagination)

The API returns at most 50 results per page. `nextPageToken` is a bookmark for the next page; when it's missing, there are no more videos.
We keep the newest 300 videos per channel.

In [15]:
def get_video_ids(playlist_id, max_videos=300):
    ids, page_token = [], None
    while len(ids) < max_videos:
        resp = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=page_token,
        ).execute()
        ids += [item["contentDetails"]["videoId"] for item in resp["items"]]
        page_token = resp.get("nextPageToken")
        if not page_token:          # no more pages
            break
    return ids[:max_videos]

video_ids = {}
for row in channels.itertuples():
    video_ids[row.channel_id] = get_video_ids(row.uploads_playlist)
    print(row.channel_name, len(video_ids[row.channel_id]))

MaddyEats 300
CarryisLive 300
lastMinute 300
Thugesh 300
Scout 300
T-Series 300
RON GAMING 300
MortaL 300
Ujjwal 300
Sumit 007 300
YesSmartyPie 300
GamerBlox 147
Technical Guruji 300
Kaash Plays 300
Anshu Bisht 300
Apna College 300
Sevou 300
Akash Musale 85
HISTORY 300
PAYAL GAMING 300
Trakin Tech 300
Mysterious Dunia 300
BACHU 242
Levinho 300
CampusX 300
NotGamerFleet 300
Antaryami Gaming 300
Wake'N'Bite 300
Jaydeep Plays 300
Techno Gamerz 300


## Video details

`videos.list` returns details for 50 videos per call. `.get(..., default)` handles fields some videos don't have (no tags, hidden like counts).

`likes` and `comments` are collected **for reference only**. They happen *after* a video gets views, so using them to predict popularity would be data leakage.

In [16]:
rows = []
for channel_id, ids in video_ids.items():
    for i in range(0, len(ids), 50):                 # 50 IDs per call
        resp = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(ids[i:i + 50]),
        ).execute()
        for v in resp["items"]:
            rows.append({
                "video_id": v["id"],
                "channel_id": channel_id,
                "title": v["snippet"]["title"],
                "description": v["snippet"].get("description", ""),
                "tags": "|".join(v["snippet"].get("tags", [])),
                "category_id": v["snippet"]["categoryId"],
                "published_at": v["snippet"]["publishedAt"],
                "duration": v["contentDetails"]["duration"],
                "live": v["snippet"].get("liveBroadcastContent", "none"),
                "views": int(v["statistics"].get("viewCount", 0)),
                "likes": int(v["statistics"].get("likeCount", 0)),
                "comments": int(v["statistics"].get("commentCount", 0)),
            })

videos = pd.DataFrame(rows)
videos.shape

(8574, 12)

In [17]:
videos.head()

,video_id,channel_id,title,description,tags,category_id,published_at,duration,live,views,likes,comments
0,WhpNOrxsDR4,UCgcUFVynuKwaIZLJkjhzOVg,"MUKBANG WITH BFF AND BRO🔥😋 SPICY ALOO BIRYANI,...",Aloo Biryani Recipe \n\nhttps://youtu.be/j8dL5...,,22,2026-10-03T09:01:19Z,PT14M28S,none,87472,1331,72
1,s9vP2PaoPkg,UCgcUFVynuKwaIZLJkjhzOVg,"VIRAL SPICY CHEESY BULDAK NOODLES ONION WRAP, ...",VIRAL SPICY CHEESY BULDAK NOODLES ONION WRAP R...,,22,2026-09-30T09:38:51Z,PT16M17S,none,252525,2636,127
2,S0kxdLYDB1Q,UCgcUFVynuKwaIZLJkjhzOVg,"CHICKEN BIRYANI, PANEER BIRYANI AND SPICY NARG...",Pineapple Fried Rice Recipe \n\nhttps://youtu....,,22,2026-09-26T09:59:56Z,PT16M8S,none,248429,2523,138
3,HuL5YuhU9bk,UCgcUFVynuKwaIZLJkjhzOVg,"SPICY CHICKEN CHOPPER FRIED RICE, JAPANESE FLU...",VIRAL JAPANESE FLUFFY CLOUD EGG OMELETTE RECIP...,,22,2026-09-23T09:47:22Z,PT16M51S,none,367851,3727,211
4,Tbo47bpCHvE,UCgcUFVynuKwaIZLJkjhzOVg,"CHICKEN LOLLIPOP BIRYANI, SPICY CHICKEN GRAVY ...",CHICKEN LOLLIPOP BIRYANI RECIPE \n\nhttps://yo...,,22,2026-09-18T10:47:06Z,PT16M54S,none,368597,3477,152


## Redact leaked secrets

Some creators paste code into their video descriptions, including API keys. Two coding-tutorial descriptions in this dataset
contained an OpenAI key and a LangSmith token (GitHub's push protection caught them).
Republishing someone else's key would spread the leak, so anything that looks like a secret is replaced with `[REDACTED]` before saving.

In [ ]:
import re

# Secret patterns: OpenAI, LangSmith, Google, GitHub, Hugging Face, Slack
SECRET_PATTERN = re.compile(
    r"sk-[A-Za-z0-9_\-]{20,}|lsv2_(?:pt|sk)_[A-Za-z0-9_]{20,}|AIza[0-9A-Za-z_\-]{30,}"
    r"|gh[pousr]_[A-Za-z0-9]{30,}|hf_[A-Za-z0-9]{30,}|xox[baprs]-[A-Za-z0-9-]{10,}"
)


def redact_secrets(text):
    return SECRET_PATTERN.sub("[REDACTED]", text) if isinstance(text, str) else text


for col in ["title", "description", "tags"]:
    videos[col] = videos[col].apply(redact_secrets)

print("Secrets left:", sum(videos[c].fillna("").str.contains(SECRET_PATTERN).sum() for c in ["title", "description", "tags"]))

## Save

Collect once, then work offline: every later notebook reads these CSVs, with no further API calls.

In [18]:
videos.to_csv("videos_raw.csv", index=False)
channels.to_csv("channels.csv", index=False)
print("Duplicate video IDs:", videos["video_id"].duplicated().sum())

Duplicate video IDs: 0
